# Módulo 06 · Sistemas de recomendación — Soluciones de los ejercicios

Este notebook es **autocontenido**: reproduce la preparación del notebook de la práctica (mismos datos simulados, misma semilla, mismos modelos) y después resuelve los cuatro ejercicios del final:

1. Arranque en frío para usuarios y películas nuevos (estrategia híbrida).
2. Efecto de la regularización `lam` en ALS.
3. Filtrado colaborativo usuario-usuario frente a ítem-ítem (calidad y coste).
4. Repetir la comparación con **MovieLens 100K** (con plan B offline si no hay conexión).

> Los números de las interpretaciones corresponden a la ejecución con la semilla del notebook. Si cambias algo, revisa que las conclusiones sigan cuadrando.

## 0. Preparación (idéntica al notebook de la práctica)

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(1)

n_usuarios, n_peliculas, k_real = 600, 300, 4
U_real = rng.normal(0, 1, (n_usuarios, k_real))
V_real = rng.normal(0, 1, (n_peliculas, k_real))
sesgo_u = rng.normal(0, 0.4, n_usuarios)
sesgo_p = rng.normal(0, 0.6, n_peliculas)
nota_latente = 3.4 + sesgo_u[:, None] + sesgo_p[None, :] + 0.5 * U_real @ V_real.T + rng.normal(0, 0.4, (n_usuarios, n_peliculas))
notas_completas = np.clip(np.round(nota_latente), 1, 5)

popularidad = rng.pareto(1.5, n_peliculas) + 1
prob_ver = popularidad[None, :] / popularidad.sum() * 30 * np.exp(0.3 * (nota_latente - 3))
observada = rng.random((n_usuarios, n_peliculas)) < np.clip(prob_ver, 0, 0.9)

filas, cols = np.where(observada)
valoraciones = pd.DataFrame({"usuario": filas, "pelicula": cols, "nota": notas_completas[filas, cols]})
print(f"{len(valoraciones):,} valoraciones | densidad de la matriz: {observada.mean():.1%}")

In [ ]:
test = valoraciones.groupby("usuario", group_keys=False).sample(frac=0.2, random_state=0)
train = valoraciones.drop(test.index)

R = np.full((n_usuarios, n_peliculas), np.nan)
R[train["usuario"], train["pelicula"]] = train["nota"]
media_global = np.nanmean(R)

def rmse(pred, df=test):
    return np.sqrt(np.mean((pred[df["usuario"], df["pelicula"]] - df["nota"]) ** 2))

def precision_at_k(puntuaciones, k=5, umbral=5, df=None):
    """Igual que en el notebook, pero admite otro conjunto de test con `df`."""
    df = test if df is None else df
    precisiones = []
    for u, grupo in df.groupby("usuario"):
        if len(grupo) < k:
            continue
        top = grupo.assign(p=puntuaciones[u, grupo["pelicula"]]).nlargest(k, "p")
        precisiones.append((top["nota"] >= umbral).mean())
    return np.mean(precisiones)

pred_global = np.full((n_usuarios, n_peliculas), media_global)

def sesgos(R, lam=5, iters=10):
    bu, bi = np.zeros(R.shape[0]), np.zeros(R.shape[1])
    mask = ~np.isnan(R)
    Rc = np.where(mask, R - media_global, 0)
    for _ in range(iters):
        bi = ((Rc - bu[:, None]) * mask).sum(0) / (mask.sum(0) + lam)
        bu = ((Rc - bi[None, :]) * mask).sum(1) / (mask.sum(1) + lam)
    return bu, bi

bu, bi = sesgos(R)
pred_sesgos = media_global + bu[:, None] + bi[None, :]
pred_popular = np.tile(np.bincount(train["pelicula"], minlength=n_peliculas), (n_usuarios, 1)).astype(float)

def filtrado_item_item(R, k_vecinos=30, encogimiento=10):
    mask = ~np.isnan(R)
    residuos = np.where(mask, R - pred_sesgos, 0)
    co = mask.T.astype(float) @ mask.astype(float)
    normas = np.sqrt((residuos ** 2).sum(0))
    sim = (residuos.T @ residuos) / (np.outer(normas, normas) + 1e-9)
    sim *= co / (co + encogimiento)
    np.fill_diagonal(sim, 0)
    corte = -np.sort(-sim, axis=1)[:, k_vecinos - 1:k_vecinos]
    sim = np.where(sim >= corte, sim, 0)
    num = residuos @ sim.T
    den = mask.astype(float) @ np.abs(sim).T
    return pred_sesgos + np.divide(num, den, out=np.zeros_like(num), where=den > 1e-9), sim

pred_item, sim_items = filtrado_item_item(R)

def als(R, k=8, lam=8.0, iters=15, rng=rng):
    mask = ~np.isnan(R)
    Rc = np.where(mask, R - pred_sesgos, 0)
    P = rng.normal(0, 0.1, (R.shape[0], k))
    Q = rng.normal(0, 0.1, (R.shape[1], k))
    I = lam * np.eye(k)
    for _ in range(iters):
        for u in range(R.shape[0]):
            idx = mask[u]
            P[u] = np.linalg.solve(Q[idx].T @ Q[idx] + I, Q[idx].T @ Rc[u, idx])
        for i in range(R.shape[1]):
            idx = mask[:, i]
            Q[i] = np.linalg.solve(P[idx].T @ P[idx] + I, P[idx].T @ Rc[idx, i])
    return pred_sesgos + P @ Q.T, P, Q

pred_als, P, Q = als(R)

resultados = pd.DataFrame({
    "RMSE": [np.nan, rmse(pred_global), rmse(pred_sesgos), rmse(pred_item), rmse(pred_als)],
    "precisión@5": [precision_at_k(pred_popular), precision_at_k(pred_global + rng.normal(0, 1e-6, pred_global.shape)),
                     precision_at_k(pred_sesgos), precision_at_k(pred_item), precision_at_k(pred_als)],
}, index=["Popularidad", "Media global (≈ aleatorio)", "Sesgos", "Ítem-ítem", "ALS (k=8)"])
resultados.round(3)

Es la misma tabla de la sección 6 del notebook. La usaremos como punto de comparación en los ejercicios.

---
## Ejercicio 1 · Arranque en frío

### 1a. Usuario recién registrado
Un usuario nuevo no tiene fila en $P$ ni sesgo $b_u$: los modelos colaborativos no pueden personalizar. Lo que sí tenemos son señales **no personalizadas** de cada película:

- **Popularidad** (cuántas veces se ha visto): predice bien *qué* va a ver la gente.
- **Calidad regularizada** ($b_i$ del modelo de sesgos, una "media bayesiana" que desconfía de películas con pocas notas): predice *si le gustará*.

Simulamos que **todos** los usuarios de test son nuevos y les recomendamos las mismas 10 películas. Medimos dos cosas sobre sus valoraciones de test:
- **cobertura de visionado**: fracción de usuarios que acabaron viendo al menos una de las 10 recomendadas;
- **aciertos de 5 estrellas**: fracción de usuarios que vieron y puntuaron con 5 alguna de las 10.

In [ ]:
conteo = np.bincount(train["pelicula"], minlength=n_peliculas).astype(float)

def evaluar_lista_fija(lista, df=test):
    lista = set(int(i) for i in lista)
    vistas = df[df["pelicula"].isin(lista)]
    cobertura = vistas["usuario"].nunique() / df["usuario"].nunique()
    aciertos5 = vistas.loc[vistas["nota"] >= 5, "usuario"].nunique() / df["usuario"].nunique()
    nota_media = vistas["nota"].mean()
    return cobertura, aciertos5, nota_media

# Híbrido sencillo: combinar popularidad (en log) y calidad, ambas estandarizadas
z = lambda x: (x - x.mean()) / x.std()
listas = {
    "Más populares": np.argsort(-conteo)[:10],
    "Mejor valoradas (b_i regularizado)": np.argsort(-bi)[:10],
    "Media bruta (sin regularizar)": np.argsort(-np.nan_to_num(np.nanmean(R, axis=0), nan=0))[:10],
    "Híbrido 50 % popularidad + 50 % calidad": np.argsort(-(0.5 * z(np.log1p(conteo)) + 0.5 * z(bi)))[:10],
}
frio_usuario = pd.DataFrame(
    [evaluar_lista_fija(l) for l in listas.values()],
    index=listas.keys(), columns=["cobertura de visionado", "aciertos de 5★", "nota media de lo visto"],
)
frio_usuario["valoraciones (train) de la lista"] = [int(conteo[l].sum()) for l in listas.values()]
frio_usuario.round(3)

**Interpretación.**
- La lista de **más populares** es la que más usuarios acaban viendo (cobertura 0,815), pero lo que ven les gusta solo regular (nota media 3,90).
- La lista por **$b_i$ regularizado** tiene una nota media de lo visto mucho más alta (4,59), pero solo la ve el 36 % de los usuarios: son buenas películas, pero de nicho.
- La **media bruta** sin regularizar es la peor opción: se llena de películas con muy pocas notas (706 valoraciones en total entre las 10), tiene la menor cobertura (0,287) y el menor porcentaje de usuarios con un acierto de 5★ (0,215).
- El **híbrido** se queda con lo mejor de ambos: cobertura alta (0,717), nota media 4,15 y el mayor porcentaje de usuarios con un acierto de 5★ (0,418). Es una recomendación por defecto razonable para un usuario sin historial.

**Estrategia completa para un usuario nuevo (respuesta modelo):**
1. Día 0: lista híbrida popularidad + calidad, **segmentada** si hay contexto (país, idioma, dispositivo, franja horaria, canal de captación).
2. Incorporación (*onboarding*): pedir 3–5 géneros o títulos favoritos y usarlos como valoraciones iniciales (o para elegir el segmento).
3. Exploración: reservar 1–2 huecos de la parrilla para títulos variados y aprender rápido de los clics.
4. En cuanto tenga unas pocas interacciones, calcular su vector $p_u$ **sin reentrenar todo**: con $Q$ fijo, $p_u$ es la solución de un problema de mínimos cuadrados regularizado de tamaño $k$ (el mismo paso de ALS). Ítem-ítem también funciona desde la primera película vista.

### 1b. Película recién estrenada
Una película sin valoraciones no tiene $b_i$ ni $q_i$. La solución estándar es **híbrida**: aprender una función que lleve de los **metadatos** de la película (géneros, director, reparto, sinopsis…) a sus parámetros colaborativos ($b_i$ y $q_i$), entrenada con las películas que sí tienen historial.

Simulamos unos metadatos imperfectos: los "géneros" que anota el equipo editorial son el vector latente real más ruido, y la "nota de la crítica" es la calidad real de la película más ruido. Retiramos 30 películas por completo del entrenamiento (como si fueran estrenos), entrenamos sesgos y ALS sin ellas y predecimos sus notas con tres estrategias.

In [ ]:
from sklearn.linear_model import Ridge

rng_frio = np.random.default_rng(7)
metadatos = V_real + rng_frio.normal(0, 0.7, V_real.shape)      # "géneros" anotados con ruido
critica = sesgo_p + rng_frio.normal(0, 0.4, n_peliculas)        # nota de la crítica profesional (ruidosa)
metadatos = np.column_stack([metadatos, critica])

n_vals = valoraciones.groupby("pelicula").size()
estrenos = rng_frio.choice(n_vals[n_vals >= 20].index.to_numpy(), size=30, replace=False)
calientes = np.setdiff1d(np.arange(n_peliculas), estrenos)

R_frio = R.copy()
R_frio[:, estrenos] = np.nan                                     # nadie las ha valorado aún

# Volvemos a entrenar sesgos y ALS sin los estrenos (las funciones del notebook usan las variables globales)
media_global_bak, pred_sesgos_bak = media_global, pred_sesgos
media_global = np.nanmean(R_frio)
bu_f, bi_f = sesgos(R_frio)
pred_sesgos = media_global + bu_f[:, None] + bi_f[None, :]
_, P_f, Q_f = als(R_frio, rng=np.random.default_rng(3))
mu_f = media_global
media_global, pred_sesgos = media_global_bak, pred_sesgos_bak   # restauramos

# Modelo de contenido: metadatos -> (b_i, q_i), entrenado con las películas "calientes"
Y = np.column_stack([bi_f, Q_f])
modelo_meta = Ridge(alpha=1.0).fit(metadatos[calientes], Y[calientes])
Y_hat = modelo_meta.predict(metadatos[estrenos])
bi_hat, Q_hat = Y_hat[:, 0], Y_hat[:, 1:]
# Variante con solo los géneros (sin la nota de la crítica) para estimar q_i
Q_hat_generos = Ridge(alpha=1.0).fit(metadatos[calientes, :k_real], Q_f[calientes]).predict(metadatos[estrenos, :k_real])

# Evaluación: todas las valoraciones reales de los estrenos (las de train y las de test)
ev = valoraciones[valoraciones["pelicula"].isin(estrenos)].copy()
pos = {int(p): j for j, p in enumerate(estrenos)}
j = ev["pelicula"].map(pos).to_numpy()
u_ev = ev["usuario"].to_numpy()
preds_frio = {
    "Media + b_u (sin info. de la película)": mu_f + bu_f[u_ev],
    "Media + b_u + q_i estimado (solo géneros)": mu_f + bu_f[u_ev] + np.sum(P_f[u_ev] * Q_hat_generos[j], axis=1),
    "Media + b_u + b_i estimado (metadatos)": mu_f + bu_f[u_ev] + bi_hat[j],
    "Media + b_u + b_i + q_i estimados (híbrido)": mu_f + bu_f[u_ev] + bi_hat[j] + np.sum(P_f[u_ev] * Q_hat[j], axis=1),
}
frio_pelicula = pd.Series({nombre: np.sqrt(np.mean((p - ev["nota"].to_numpy()) ** 2)) for nombre, p in preds_frio.items()},
                          name="RMSE en estrenos")
print(f"{len(ev):,} valoraciones de {len(estrenos)} estrenos")
frio_pelicula.round(3).to_frame()

**Interpretación.** Sin ninguna valoración del estreno:
- Solo con la media y el sesgo del usuario el RMSE es 1,076 (casi como la media global del notebook, 1,091).
- Los **géneros** permiten estimar $q_i$, es decir, *a quién* le gustará: el error baja a 1,020.
- La **nota de la crítica** permite estimar $b_i$, *cuánto* gustará en general: aquí es la señal que más ayuda (0,923).
- Combinando ambas (híbrido) llegamos a 0,855, mejor que el modelo de sesgos con películas conocidas (0,957), aunque todavía lejos de ALS con historial (0,746).

Moraleja: cada tipo de metadato aporta una pieza distinta. Los géneros dicen a quién va dirigida la película, pero no si es buena; para eso hacen falta señales de calidad (crítica, primeras reacciones, rendimiento en taquilla).

**Estrategia completa para una película nueva (respuesta modelo):**
1. Usar contenido/metadatos (géneros, reparto, director, *embeddings* de la sinopsis o del tráiler) para estimar $b_i$ y $q_i$, como arriba, o para recomendarla a quien vio películas parecidas en contenido.
2. Darle **exposición controlada** (un pequeño porcentaje de impresiones, o un algoritmo de *bandits*) para recoger las primeras interacciones rápidamente.
3. A medida que acumula valoraciones, ir pasando de la estimación por contenido a la colaborativa (por ejemplo, ponderando por $n_i/(n_i+\lambda)$, el mismo encogimiento que usan los sesgos).
4. Reglas de negocio: estrenos propios o con derechos limitados pueden tener una cuota editorial mínima en la portada.

---
## Ejercicio 2 · Regularización `lam` en ALS
Entrenamos ALS con $k=8$ y los valores de $\lambda$ pedidos (0,1; 1; 8; 30) más dos intermedios (0,3 y 3) para ver mejor la curva, y medimos el error en **train** y en **test**.

In [ ]:
filas_lam = []
for lam in [0.1, 0.3, 1, 3, 8, 30]:
    pred_l, P_l, Q_l = als(R, k=8, lam=lam, iters=15, rng=np.random.default_rng(0))
    filas_lam.append({
        "lam": lam,
        "RMSE train": rmse(pred_l, train),
        "RMSE test": rmse(pred_l, test),
        "precisión@5": precision_at_k(pred_l),
        "norma media de p_u": np.linalg.norm(P_l, axis=1).mean(),
    })
tabla_lam = pd.DataFrame(filas_lam).set_index("lam")
tabla_lam.round(3)

In [ ]:
ax = tabla_lam[["RMSE train", "RMSE test"]].plot(marker="o", logx=True, figsize=(6, 3.5))
ax.axhline(rmse(pred_sesgos), ls="--", c="grey", label="solo sesgos (test)")
ax.set_xlabel("λ (escala log)"); ax.set_ylabel("RMSE"); ax.set_title("Sobreajuste frente a regularización en ALS")
ax.legend(); plt.tight_layout(); plt.show()

**Interpretación (sobreajuste).**
- Con **λ = 0,1** el error de **train** es bajísimo (0,357) pero el de **test** sube a 0,869: muchos usuarios tienen pocas decenas de valoraciones y 8 factores libres por usuario bastan para "memorizar" parte del ruido. Los vectores $p_u$ crecen (la norma media pasa de 0,63 con λ = 8 a 2,29). Es **sobreajuste**: la mayor distancia entre train y test de toda la tabla.
- Entre **λ = 1 y λ = 3** está el mínimo del error de test (≈ 0,68) y la mejor precisión@5 (≈ 0,41). Fíjate en que el valor por defecto del notebook (λ = 8) **no es el óptimo** en estos datos: compensa ajustar hiperparámetros.
- Con **λ = 8** ya hay algo de exceso de regularización (test 0,746) y con **λ = 30** el modelo está **infraajustado**: los factores quedan casi a cero (norma 0,08), los errores de train (0,900) y test (0,950) son altos y parecidos, y el resultado es prácticamente el del modelo de solo sesgos (0,957).
- La penalización $\lambda(\lVert p_u\rVert^2 + \lVert q_i\rVert^2)$ encoge los vectores hacia cero, es decir, hacia la predicción de sesgos, sobre todo los de usuarios y películas con pocos datos. Regla práctica: λ (y $k$) se eligen con un conjunto de **validación**, nunca mirando el error de train.

---
## Ejercicio 3 · Usuario-usuario frente a ítem-ítem
La implementación es la "transpuesta" de la del notebook: la similitud se calcula entre **filas** (usuarios) y la predicción es la media ponderada de las desviaciones de los vecinos que **sí** vieron esa película.

In [ ]:
def filtrado_usuario_usuario(R, k_vecinos=30, encogimiento=10):
    mask = ~np.isnan(R)
    residuos = np.where(mask, R - pred_sesgos, 0)
    co = mask.astype(float) @ mask.T.astype(float)                   # películas en común entre usuarios
    normas = np.sqrt((residuos ** 2).sum(1))
    sim = (residuos @ residuos.T) / (np.outer(normas, normas) + 1e-9)
    sim *= co / (co + encogimiento)
    np.fill_diagonal(sim, 0)
    corte = -np.sort(-sim, axis=1)[:, k_vecinos - 1:k_vecinos]
    sim = np.where(sim >= corte, sim, 0)                             # k vecinos por usuario
    num = sim @ residuos                                             # (usuarios × usuarios) · (usuarios × películas)
    den = np.abs(sim) @ mask.astype(float)
    return pred_sesgos + np.divide(num, den, out=np.zeros_like(num), where=den > 1e-9), sim

t0 = time.perf_counter(); pred_uu, sim_usuarios = filtrado_usuario_usuario(R); t_uu = time.perf_counter() - t0
t0 = time.perf_counter(); pred_ii, _ = filtrado_item_item(R); t_ii = time.perf_counter() - t0

comparacion_cf = pd.DataFrame({
    "RMSE": [rmse(pred_uu), rmse(pred_ii)],
    "precisión@5": [precision_at_k(pred_uu), precision_at_k(pred_ii)],
    "tamaño matriz de similitud": [f"{sim_usuarios.shape[0]}×{sim_usuarios.shape[1]}", f"{sim_items.shape[0]}×{sim_items.shape[1]}"],
    "segundos": [t_uu, t_ii],
}, index=["Usuario-usuario", "Ítem-ítem"])
comparacion_cf.round(3)

**Calidad.** En estos datos usuario-usuario queda por detrás de ítem-ítem en RMSE (0,856 frente a 0,813) y prácticamente empata en precisión@5 (0,388 frente a 0,390). En train, cada película tiene de media unas 55 valoraciones y cada usuario unas 28: las similitudes entre películas se estiman con más datos y son más fiables. Con matrices tan pequeñas ambos tardan décimas de segundo; la diferencia de coste aparece a gran escala.

**Coste con 10 millones de usuarios y 50.000 productos.** Hacemos las cuentas:

In [ ]:
n_u_grande, n_i_grande = 10_000_000, 50_000
bytes_float = 4
print(f"Matriz usuario-usuario: {n_u_grande**2:.1e} pares -> {n_u_grande**2 * bytes_float / 1e12:,.0f} TB en float32")
print(f"Matriz ítem-ítem:       {n_i_grande**2:.1e} pares -> {n_i_grande**2 * bytes_float / 1e9:,.0f} GB en float32")
print(f"Guardando solo 50 vecinos por fila: usuarios {n_u_grande*50*8/1e9:,.0f} GB | ítems {n_i_grande*50*8/1e6:,.0f} MB (id + similitud)")
print(f"Ratio de pares: {n_u_grande**2 / n_i_grande**2:,.0f} veces más pares en usuario-usuario")

**Respuesta.** **Ítem-ítem es mucho más barato**: hay $4\cdot 10^4$ veces más pares de usuarios que de productos. Además:
- Las similitudes entre productos son **estables** (dos películas parecidas lo siguen siendo mañana), así que se precalculan por lotes (por ejemplo, cada noche) y se sirven con una consulta rápida. Las de usuarios cambian con cada clic.
- Un usuario nuevo con una sola interacción ya recibe recomendaciones ítem-ítem ("porque viste X"); en usuario-usuario necesitaría historial suficiente para encontrar vecinos.
- Son **explicables**: "te recomendamos Y porque viste X".

Usuario-usuario puede tener sentido cuando hay pocos usuarios y un catálogo enorme o muy cambiante (por ejemplo, noticias), o para descubrir "gente como tú". A gran escala, ambos se implementan con búsqueda aproximada de vecinos (ANN) en lugar de matrices densas.

---
## Ejercicio 4 · Datos reales: MovieLens 100K
Orden de preferencia para obtener los datos:
1. un archivo `u.data` local (si lo has descargado a mano junto a este notebook);
2. la descarga desde GroupLens;
3. **plan B offline**: si no hay conexión, generamos un conjunto **simulado** con las mismas dimensiones que MovieLens 100K (943 usuarios, 1.682 películas, unas 100.000 notas) para que el notebook se ejecute siempre. En ese caso los números no son los de MovieLens: el mensaje lo indica claramente.

In [ ]:
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
fuente = None
try:
    if Path("u.data").exists():
        df = pd.read_csv("u.data", sep="\t", names=["usuario", "pelicula", "nota", "ts"])
        fuente = "MovieLens 100K (archivo local)"
    else:
        df = pd.read_csv(url, sep="\t", names=["usuario", "pelicula", "nota", "ts"])
        fuente = "MovieLens 100K (descargado de GroupLens)"
    df["usuario"] -= 1                                                 # los IDs empiezan en 1
    df["pelicula"] -= 1
except Exception as e:
    print(f"⚠️ No se ha podido descargar MovieLens 100K ({type(e).__name__}).")
    print("   Plan B: datos SIMULADOS con el tamaño de MovieLens 100K. Los resultados NO son los de MovieLens.")
    print("   Para usar los datos reales, descarga u.data de", url, "y colócalo junto al notebook.")
    rng_ml = np.random.default_rng(100)
    n_u, n_i = 943, 1682
    U_ = rng_ml.normal(0, 1, (n_u, 6)); V_ = rng_ml.normal(0, 1, (n_i, 6))
    lat = 3.3 + rng_ml.normal(0, 0.45, n_u)[:, None] + rng_ml.normal(0, 0.6, n_i)[None, :] \
          + 0.35 * U_ @ V_.T + rng_ml.normal(0, 0.7, (n_u, n_i))
    notas_ = np.clip(np.round(lat), 1, 5)
    pop_ = rng_ml.pareto(1.2, n_i) + 1
    actividad = rng_ml.lognormal(0, 0.8, n_u)                          # usuarios más y menos activos
    p_ = pop_[None, :] / pop_.sum() * 140 * actividad[:, None] / actividad.mean() * np.exp(0.3 * (lat - 3.5))
    obs_ = rng_ml.random((n_u, n_i)) < np.clip(p_, 0, 0.95)
    f_, c_ = np.where(obs_)
    df = pd.DataFrame({"usuario": f_, "pelicula": c_, "nota": notas_[f_, c_], "ts": 0})
    fuente = "SIMULADO (plan B offline, tamaño MovieLens 100K)"

n_u, n_i = df["usuario"].max() + 1, df["pelicula"].max() + 1
print(f"Fuente: {fuente}")
print(f"{len(df):,} valoraciones | {n_u} usuarios × {n_i} películas | densidad {len(df) / (n_u * n_i):.1%}")
print("Distribución de notas:", df["nota"].value_counts(normalize=True).sort_index().round(3).to_dict())

Las funciones del notebook dependen de variables globales (`test`, `media_global`, `pred_sesgos`). Para reutilizarlas sin sorpresas, las **parametrizamos**: cada una recibe lo que necesita. La lógica es exactamente la misma.

In [ ]:
def ajustar_sesgos(R, mu, lam=5, iters=10):
    mask = ~np.isnan(R)
    bu_, bi_ = np.zeros(R.shape[0]), np.zeros(R.shape[1])
    Rc = np.where(mask, R - mu, 0)
    for _ in range(iters):
        bi_ = ((Rc - bu_[:, None]) * mask).sum(0) / (mask.sum(0) + lam)
        bu_ = ((Rc - bi_[None, :]) * mask).sum(1) / (mask.sum(1) + lam)
    return bu_, bi_

def item_item_general(R, base, k_vecinos=30, encogimiento=10):
    mask = ~np.isnan(R)
    residuos = np.where(mask, R - base, 0)
    co = mask.T.astype(float) @ mask.astype(float)
    normas = np.sqrt((residuos ** 2).sum(0))
    sim = (residuos.T @ residuos) / (np.outer(normas, normas) + 1e-9)
    sim *= co / (co + encogimiento)
    np.fill_diagonal(sim, 0)
    corte = -np.sort(-sim, axis=1)[:, k_vecinos - 1:k_vecinos]
    sim = np.where(sim >= corte, sim, 0)
    num = residuos @ sim.T
    den = mask.astype(float) @ np.abs(sim).T
    return base + np.divide(num, den, out=np.zeros_like(num), where=den > 1e-9)

def als_general(R, base, k=8, lam=8.0, iters=15, semilla=0):
    g = np.random.default_rng(semilla)
    mask = ~np.isnan(R)
    Rc = np.where(mask, R - base, 0)
    P_ = g.normal(0, 0.1, (R.shape[0], k)); Q_ = g.normal(0, 0.1, (R.shape[1], k))
    I = lam * np.eye(k)
    for _ in range(iters):
        for u in range(R.shape[0]):
            idx = mask[u]
            P_[u] = np.linalg.solve(Q_[idx].T @ Q_[idx] + I, Q_[idx].T @ Rc[u, idx])
        for i in range(R.shape[1]):
            idx = mask[:, i]
            Q_[i] = np.linalg.solve(P_[idx].T @ P_[idx] + I, P_[idx].T @ Rc[idx, i])
    return base + P_ @ Q_.T

In [ ]:
test_ml = df.groupby("usuario", group_keys=False).sample(frac=0.2, random_state=0)
train_ml = df.drop(test_ml.index)
R_ml = np.full((n_u, n_i), np.nan)
R_ml[train_ml["usuario"], train_ml["pelicula"]] = train_ml["nota"]
mu_ml = np.nanmean(R_ml)

t = {}
t0 = time.perf_counter()
bu_ml, bi_ml = ajustar_sesgos(R_ml, mu_ml)
base_ml = mu_ml + bu_ml[:, None] + bi_ml[None, :]
t["Sesgos"] = time.perf_counter() - t0
t0 = time.perf_counter(); pred_item_ml = item_item_general(R_ml, base_ml); t["Ítem-ítem"] = time.perf_counter() - t0
t0 = time.perf_counter(); pred_als_ml = als_general(R_ml, base_ml, k=8, lam=8.0); t["ALS (k=8)"] = time.perf_counter() - t0
pred_pop_ml = np.tile(np.bincount(train_ml["pelicula"], minlength=n_i), (n_u, 1)).astype(float)
pred_glob_ml = np.full((n_u, n_i), mu_ml) + np.random.default_rng(0).normal(0, 1e-6, (n_u, n_i))

modelos_ml = {"Popularidad": pred_pop_ml, "Media global (≈ aleatorio)": pred_glob_ml,
              "Sesgos": base_ml, "Ítem-ítem": pred_item_ml, "ALS (k=8)": pred_als_ml}
resultados_ml = pd.DataFrame({
    "RMSE": [np.nan if n == "Popularidad" else rmse(p, test_ml) for n, p in modelos_ml.items()],
    "precisión@5": [precision_at_k(p, df=test_ml) for p in modelos_ml.values()],
    "segundos de entrenamiento": [t.get(n, 0.0) for n in modelos_ml],
}, index=modelos_ml.keys())
print("Fuente:", fuente)
resultados_ml.round(3)

Como en el notebook, probamos también varios valores de $k$ y $\lambda$ para ALS (en un proyecto real, esto se haría con un conjunto de **validación** separado del test; aquí lo simplificamos).

In [ ]:
rejilla = []
for k_ in [4, 8, 16]:
    for lam_ in [5, 15]:
        p_ = als_general(R_ml, base_ml, k=k_, lam=lam_, iters=8)
        rejilla.append({"k": k_, "lam": lam_, "RMSE test": rmse(p_, test_ml)})
pd.DataFrame(rejilla).pivot(index="k", columns="lam", values="RMSE test").round(3)

In [ ]:
print("Fuente de los datos:", fuente)
mejor_rmse = resultados_ml["RMSE"].idxmin()
mejor_prec = resultados_ml["precisión@5"].idxmax()
print(f"Mejor RMSE: {mejor_rmse} ({resultados_ml.loc[mejor_rmse, 'RMSE']:.3f}) | "
      f"mejora frente a sesgos: {1 - resultados_ml.loc[mejor_rmse, 'RMSE'] / resultados_ml.loc['Sesgos', 'RMSE']:.1%}")
print(f"Mejor precisión@5: {mejor_prec} ({resultados_ml.loc[mejor_prec, 'precisión@5']:.3f})")

**Interpretación.** Esta celda se ha ejecutado con el **plan B** (datos simulados del tamaño de MovieLens 100K: 100.238 notas, densidad 6,3 %) porque en el entorno de preparación no había acceso a GroupLens. Con esos datos:
- El orden es el mismo que en el notebook: media global (1,117) > sesgos (0,958) > ítem-ítem (0,911) > ALS (0,817) en RMSE, y la precisión@5 mejora en el mismo orden (0,460 → 0,499 → 0,555).
- La popularidad vuelve a quedar al nivel del azar en precisión@5 (0,275 frente a 0,285): recuerda que la métrica solo reordena películas que el usuario **ya vio**.
- En la rejilla, la mejor combinación es $k = 8$, $\lambda = 5$ (0,808); con $k = 16$ y poca regularización el error empeora (0,844).

**Qué esperar con MovieLens real.** Los datos simulados tienen una estructura latente más limpia que los reales, así que exageran la ventaja de ALS. Las referencias publicadas sobre MovieLens 100K (por ejemplo, la tabla de la librería *Surprise*, con validación cruzada de 5 particiones) dan 0,944 de RMSE al modelo de sesgos, 0,934 a SVD, 0,931 a kNN con sesgos y 1,518 a un predictor aleatorio. Es decir: con datos reales **el modelo de sesgos se lleva la mayor parte de la mejora y los modelos colaborativos añaden solo un 1–2 % más en RMSE** (en el ranking la mejora puede ser mayor). Si tus números se parecen, tu implementación es correcta. Si ALS sale peor que los sesgos, sube `lam` (sobreajuste, ejercicio 2).

**Para el caso práctico (respuesta modelo de la recomendación de despliegue):**

| Criterio | Sesgos | Ítem-ítem | ALS |
|---|---|---|---|
| Precisión (RMSE / precisión@5) | Buena referencia | Mejor | Mejor (similar o algo superior) |
| Coste de entrenamiento | Mínimo | Matriz ítems × ítems, precalculable | Iterativo, paralelizable |
| Coste de servir | Trivial | Consulta de vecinos | Producto escalar $p_u^\top q_i$ (ANN) |
| Explicabilidad | "Es muy popular y bien valorada" | "Porque viste X" | Baja (factores sin nombre) |
| Usuario con 1–2 interacciones | Sí (no personaliza) | Sí | Requiere recalcular $p_u$ |

Una recomendación defendible: **ALS (o ítem-ítem si la explicabilidad pesa) para la parrilla principal, con el modelo de sesgos/popularidad como respaldo para el arranque en frío**, y la decisión final validada con una prueba A/B cuya métrica principal sea de negocio (minutos vistos o retención a 30 días), no el RMSE.